In [ ]:
import os
parent_dir = '/liu1/Son/Projects/PD1/results_local/peptide_1_1' #replace with different path if needed
for root, dirs, files in os.walk(parent_dir):
    for file in files:
        if file.endswith('.sdf'):
            !obabel -i sdf {os.path.join(root, file)} -o mol2 -O {os.path.join(root, file.replace('.sdf', '.mol2'))} -l 1 -h
            !python /liu1/Son/ProDock-dev-son/Preparation_script/mol2_uniqueatoms.py {os.path.join(root, file.replace('.sdf', '.mol2'))} {os.path.join(root, file[:-4]+'_unique.mol2')}
            !perl ./sort_mol2_bonds.pl {os.path.join(root, file[:-4]+'_unique.mol2')} {os.path.join(root, file.replace('.sdf', '.mol2'))}
            !obabel -i mol2 {os.path.join(root, file.replace('.sdf', '.mol2'))} -o mol2 -O {os.path.join(root, file.replace('.sdf', '.mol2'))}

In [ ]:
import os
parent_dir = '/home/lelaihoangson/Documents/Workspace/Son/Projects/PD1/0_100'
for root, dirs, files in os.walk(parent_dir):
    for dir in dirs:
        if dir.startswith('peptide_'):
            child_dir = os.path.join(root, dir)
            os.chdir(child_dir)
            !python cgenff_charmm2gmx_py3_nx2.py LIG1 {dir}.mol2 {dir}.str charmm36-jul2022.ff
            !gmx editconf -f lig1_ini.pdb -o lig1.gro
            !gmx pdb2gmx -f PD1.pdb -o PD1.gro -water tip3p -ignh -ff charmm36-jul2022
            with open('lig1.gro', 'r') as f:
                lig_lines = f.readlines()

            # Read PD1.gro
            with open('PD1.gro', 'r') as f:
                pd_lines = f.readlines()

            # Parse the number of atoms from the second line of each .gro file
            lig_count = int(lig_lines[1].strip().split()[0])
            pd_count  = int(pd_lines[1].strip().split()[0])
            new_count = lig_count + pd_count

            # Update the atom count in PD1.gro (second line)
            pd_lines[1] = f"{new_count}\n"

            # Extract the atom records from lig1.gro (skip first 2 lines and last line)
            lig_atoms = lig_lines[2:-1]

            # Insert those atom lines into PD1.gro just before its last line
            pd_body = pd_lines[:-1] + lig_atoms + [pd_lines[-1]]

            # Write the updated contents back to PD1.gro
            with open('PD1.gro', 'w') as f:
                f.writelines(pd_body)

            # ---- Modify topol.top ----

            # Read the topology file
            with open('topol.top', 'r') as f:
                topo_lines = f.readlines()

            # Open topol.top for writing updates in place
            with open('topol.top', 'w') as f:
                for line in topo_lines:
                    f.write(line)
                    # After the main force field include, add the ligand parameters
                    if line.strip() == '#include "./charmm36-jul2022.ff/forcefield.itp"':
                        f.write('#include "lig1.prm"\n')
                        f.write('#include "lig1.itp"\n')
                # Finally, append the LIG1 entry at the end of the [ molecules ] section
                f.write('LIG1                1\n')
            !gmx editconf -f PD1.gro -o PD1.gro -bt cubic -d 1.0
            !gmx solvate -cp PD1.gro -cs spc216.gro -p topol.top -o PD1.gro
            !gmx grompp -f ions.mdp -c PD1.gro -p topol.top -o ions.tpr -maxwarn 10
            !echo 15 | gmx genion -s ions.tpr -o PD1.gro -p topol.top -pname NA -nname CL -conc 0.15
            !gmx grompp -f em.mdp -c PD1.gro -p topol.top -o em.tpr -maxwarn 10
            !gmx mdrun -v -deffnm em
            !echo -e "0 & ! a H*\nq" | gmx make_ndx -f lig1.gro -o index_lig1.ndx
            !echo 3 | gmx genrestr -f lig1.gro -n index_lig1.ndx -o posre_lig1.itp -fc 1000 1000 1000
            with open('topol.top', 'r') as f:
                topo_lines = f.readlines()
            with open('topol.top', 'w') as f:
                for line in topo_lines:
                    f.write(line)
                    # After the main force field include, add the ligand parameters
                    if line.strip() == '; Include water topology':
                        f.write('#ifdef POSRES\n')
                        f.write('#include "posre_lig1.itp"\n')
                        f.write('#endif\n\n')
                        f.write(line)
            !echo -e "1 | 13\nq" |gmx make_ndx -f em.gro -o index.ndx

In [ ]:
import MDAnalysis as mda
from MDAnalysis.analysis import rms, align
# import nglview as nv

import warnings
warnings.filterwarnings('ignore')
from matplotlib.ticker import MultipleLocator, AutoMinorLocator
import pandas as pd


# suppress some MDAnalysis warnings about writing PDB files
import os
import matplotlib.pyplot as plt
%matplotlib inline
from MDAnalysisTests.datafiles import GRO, XTC
from MDAnalysis.analysis.dihedrals import Ramachandran,Dihedral
import os
import numpy as np
import prolif as plf
from MDAnalysis.topology.guessers import guess_types
from prolif.plotting.barcode import Barcode

parent_dir = "/home/lelaihoangson/Documents/Workspace/Son/Projects/PD1/MD"
with os.scandir(parent_dir) as it:
    direct_folders = [entry.name
                      for entry in it
                      if entry.is_dir() and str(entry.name) != "peptide_1_1" and str(entry.name) != "slurm"]
for i in direct_folders:
    try:
        os.chdir(f'{parent_dir}/{i}')
        !echo 1 0 | gmx trjconv -s md_0_100.tpr -f md_0_100.xtc -o md_0_100_nojump.xtc -center -ur compact -pbc nojump
        !echo 1 0 | gmx trjconv -s md_0_100.tpr -f md_0_100_nojump.xtc -o md_0_100_center.xtc -center -ur compact -pbc mol
        !echo 1 0 | gmx trjconv -s md_0_100.tpr -f md_0_100_center.xtc -o md_0_100_fit.xtc -fit rot+trans
        !gmx genconf -f md_0_100.gro -o md_0_100_renumber.gro -renumber
        for j in ["md_0_100_nojump.xtc","md_0_100_center.xtc"]:
            file_path = f"{parent_dir}/{i}/{j}"
            if os.path.exists(file_path):
                os.remove(file_path)
                print(f"{file_path} has been deleted.")
            else:
                print(f"{file_path} does not exist.")
    except Exception as e:
        break

In [ ]:
parent_dir = "/home/lelaihoangson/Documents/Workspace/Son/Projects/PD1/MD/0_100" #replace with different path if needed
for i in direct_folders:
    os.chdir(f'{parent_dir}/{i}')
    u = mda.Universe('md_0_100.gro', 'md_0_100_fit.xtc')

    u.add_TopologyAttr('elements', guess_types(u.atoms.names))
    # create selections for both protein components
    peptide_1_calpha = "C4 C10 C11 C16 C21 C27 C28 C33 C37 C40 C51"
    peptide_2_calpha = "C4 C10 C11 C16 C21 C27 C28 C33 C36 C47"
    peptide_3_calpha = "C4 C10 C11 C16 C21 C27 C28 C33 C36 C47 C70"

    if i.startswith("peptide_1"):
        ligand_selection = u.select_atoms("resname LIG1")
        average = align.AverageStructure(u, u, select='protein or resname LIG1', ref_frame=0).run()
        ref = average.results.universe
        aligner = align.AlignTraj(u, ref, select='protein or resname LIG1', in_memory=True).run()
        c_alphas_ligand = u.select_atoms('resname LIG1 and name C4 C10 C11 C16 C21 C27 C28 C33 C37 C40 C51')
        rmsd_ligand = rms.RMSD(u,select='resname LIG1 and name C4 C10 C11 C16 C21 C27 C28 C33 C37 C40 C51',ref_frame=0).run()
        rmsd_complex = rms.RMSD(u,select='(protein and name CA) or (resname LIG1 and name C4 C10 C11 C16 C21 C27 C28 C33 C37 C40 C51)',ref_frame=0).run()
    elif i.startswith("peptide_2"):
        ligand_selection = u.select_atoms("resname LIG1")
        average = align.AverageStructure(u, u, select='protein or resname LIG1', ref_frame=0).run()
        ref = average.results.universe
        aligner = align.AlignTraj(u, ref, select='protein or resname LIG1', in_memory=True).run()
        c_alphas_ligand = u.select_atoms('resname LIG1 and name C4 C10 C11 C16 C21 C27 C28 C33 C36 C47')
        rmsd_ligand = rms.RMSD(u,select='resname LIG1 and name C4 C10 C11 C16 C21 C27 C28 C33 C36 C47',ref_frame=0).run()
        rmsd_complex = rms.RMSD(u,select='(protein and name CA) or (resname LIG1 and name C4 C10 C11 C16 C21 C27 C28 C33 C36 C47)',ref_frame=0).run()
    elif i.startswith("peptide_3"):
        ligand_selection = u.select_atoms("resname LIG1")
        average = align.AverageStructure(u, u, select='protein or resname LIG1', ref_frame=0).run()
        ref = average.results.universe
        aligner = align.AlignTraj(u, ref, select='protein or resname LIG1', in_memory=True).run()
        c_alphas_ligand = u.select_atoms('resname LIG1 and name C4 C10 C11 C16 C21 C27 C28 C33 C36 C47 C70')
        rmsd_ligand = rms.RMSD(u,select='resname LIG1 and name C4 C10 C11 C16 C21 C27 C28 C33 C36 C47 C70',ref_frame=0).run()
        rmsd_complex = rms.RMSD(u,select='(protein and name CA) or (resname LIG1 and name C4 C10 C11 C16 C21 C27 C28 C33 C36 C47 C70)',ref_frame=0).run()

    # protein_selection = u.select_atoms("protein")
    # fp = plf.Fingerprint()
    # fp.run(u.trajectory, ligand_selection, protein_selection, n_jobs=40)
    # threshold = 0
    # df = fp.to_dataframe()
    # # only keep interactions occurring more frequently than the threshold
    # above_threshold_mask = df.mean() >= threshold
    # df_filtered = df.loc[:, above_threshold_mask]
    # df_filtered.index = df_filtered.index/100
    # # plot the barcode
    # fig = Barcode(df_filtered)
    # ax = fig.display(dpi=300, xlabel='Time (ns)', figsize=(8,18))
    # ax.get_figure().savefig(f'{os.path.join(parent_dir,"all","visualization")}/interaction_fingerprint_{i}_thres{threshold}.png', dpi=300, bbox_inches='tight')
            
    c_alphas_protein = u.select_atoms('protein and name CA')
    rmsf_protein = rms.RMSF(c_alphas_protein).run()

    rmsf_ligand = rms.RMSF(c_alphas_ligand).run()

    rmsd_protein = rms.RMSD(u,select='protein and name CA',ref_frame=0).run()

    rmsf_protein_csv = pd.DataFrame({
        'Residue': c_alphas_protein.resids, 
        'RMSF': rmsf_protein.results.rmsf
    })

    rmsf_ligand_csv = pd.DataFrame({
        'Atoms': c_alphas_ligand.resids, 
        'RMSF': rmsf_ligand.results.rmsf
    })

    rmsf_protein_csv.to_csv(f'{os.path.join(parent_dir,"all","data")}/rmsf_protein_{i}.csv', index=False)
    rmsf_ligand_csv.to_csv(f'{os.path.join(parent_dir,"all","data")}/rmsf_ligand_{i}.csv', index=False)

    rmsd_csv = pd.DataFrame({
        'Time': rmsd_complex.results.rmsd[:,1]/1000,
        'RMSD_complex': rmsd_complex.results.rmsd[:,2],
        'RMSD_protein': rmsd_protein.results.rmsd[:,2],
        'RMSD_peptide': rmsd_ligand.results.rmsd[:,2]
    })
    rmsd_csv.to_csv(f'{os.path.join(parent_dir,"all","data")}/rmsd_{i}.csv', index=False)